In [2]:
"""
Global Superstore - Phase 2 | Notebook 04
Geographic Analysis
File: 02_Python/04_Geo_Analysis.py

Chay: python 02_Python/04_Geo_Analysis.py
Output: charts/chart_15_16_17.png + data/geo_summary.csv
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import sqlite3
import os
import warnings
warnings.filterwarnings("ignore")

# ── Setup ─────────────────────────────────────────────────────────────────────
DB_FILE   = r"C:\DA\global-superstore-analysis\data\superstore.db"
CHART_DIR = r"C:\DA\global-superstore-analysis\02_Python\charts"
DATA_DIR  = r"C:\DA\global-superstore-analysis\data"
os.makedirs(CHART_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "font.size":        11,
})

BLUE   = "#2E75B6"
RED    = "#C0392B"
GREEN  = "#27AE60"
ORANGE = "#E67E22"
GRAY   = "#95A5A6"

print("=" * 60)
print("GLOBAL SUPERSTORE - GEOGRAPHIC ANALYSIS")
print("=" * 60)

# ── Load data ──────────────────────────────────────────────────────────────────
conn = sqlite3.connect(DB_FILE)
df   = pd.read_sql("SELECT * FROM orders", conn)
conn.close()
print(f"\nData loaded: {df.shape[0]:,} rows | {df['country'].nunique()} countries")

# ── Tong hop theo Country ─────────────────────────────────────────────────────
country_df = df.groupby(["country", "market"]).agg(
    total_sales   = ("sales",      "sum"),
    total_profit  = ("profit",     "sum"),
    total_orders  = ("order_id",   "nunique"),
    total_qty     = ("quantity",   "sum"),
    avg_discount  = ("discount",   "mean"),
).reset_index()

country_df["margin_pct"]    = country_df["total_profit"] / country_df["total_sales"] * 100
country_df["avg_order_val"] = country_df["total_sales"]  / country_df["total_orders"]
country_df["is_loss"]       = country_df["total_profit"] < 0

print(f"\nTotal countries analyzed: {len(country_df)}")
print(f"Profitable countries    : {(~country_df['is_loss']).sum()}")
print(f"Loss-making countries   : {country_df['is_loss'].sum()}")

# ── CHART 15: Top 15 / Bottom 10 Countries ────────────────────────────────────
print("\n[Chart 15] Top & Bottom Countries by Profit...")

top15    = country_df.nlargest(15,  "total_profit")
bottom10 = country_df.nsmallest(10, "total_profit")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))

# Top 15 - xanh
bars1 = ax1.barh(top15["country"], top15["total_profit"]/1e3,
                 color=BLUE, height=0.6)
for bar, row in zip(bars1, top15.itertuples()):
    ax1.text(bar.get_width() + 1,
             bar.get_y() + bar.get_height()/2,
             f"${row.total_profit/1e3:.0f}K  ({row.margin_pct:.1f}%)",
             va="center", fontsize=8.5)
ax1.set_xlabel("Total Profit (K USD)")
ax1.set_title("Top 15 Countries by Profit", fontweight="bold", fontsize=12)
ax1.set_xlim(0, top15["total_profit"].max()/1e3 * 1.45)

# Bottom 10 - do
bars2 = ax2.barh(bottom10["country"], bottom10["total_profit"]/1e3,
                 color=RED, height=0.6)
for bar, row in zip(bars2, bottom10.itertuples()):
    ax2.text(bar.get_width() - 0.5,
             bar.get_y() + bar.get_height()/2,
             f"${row.total_profit/1e3:.1f}K  ({row.margin_pct:.1f}%)",
             va="center", ha="right", fontsize=8.5, color="white")
ax2.set_xlabel("Total Profit (K USD)")
ax2.set_title("Bottom 10 Countries (Loss-making)", fontweight="bold", fontsize=12)
ax2.axvline(0, color="black", linewidth=1)

plt.suptitle("Chart 15 — Country Profitability Analysis",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_15_country_profit.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print("   Saved: chart_15_country_profit.png")

# ── CHART 16: Market Performance ─────────────────────────────────────────────
print("[Chart 16] Market Performance...")

market_df = df.groupby("market").agg(
    total_sales  = ("sales",     "sum"),
    total_profit = ("profit",    "sum"),
    total_orders = ("order_id",  "nunique"),
    countries    = ("country",   "nunique"),
).reset_index()
market_df["margin_pct"]    = market_df["total_profit"] / market_df["total_sales"] * 100
market_df["avg_order_val"] = market_df["total_sales"]  / market_df["total_orders"]
market_df = market_df.sort_values("total_sales", ascending=False)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Sales by Market
ax = axes[0, 0]
colors_m = [BLUE if p > 0 else RED for p in market_df["total_profit"]]
bars = ax.bar(market_df["market"], market_df["total_sales"]/1e6,
              color=colors_m, width=0.6)
for bar, val in zip(bars, market_df["total_sales"]):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.05,
            f"${val/1e6:.1f}M", ha="center", fontsize=8, fontweight="bold")
ax.set_ylabel("Revenue (Million USD)")
ax.set_title("Revenue by Market", fontweight="bold")
ax.tick_params(axis="x", rotation=20)

# Profit by Market
ax = axes[0, 1]
colors_p = [GREEN if p > 0 else RED for p in market_df["total_profit"]]
bars = ax.bar(market_df["market"], market_df["total_profit"]/1e3,
              color=colors_p, width=0.6)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
for bar, val in zip(bars, market_df["total_profit"]):
    offset = 2 if val >= 0 else -8
    ax.text(bar.get_x() + bar.get_width()/2,
            val/1e3 + offset,
            f"${val/1e3:.0f}K", ha="center", fontsize=8, fontweight="bold")
ax.set_ylabel("Total Profit (K USD)")
ax.set_title("Profit by Market", fontweight="bold")
ax.tick_params(axis="x", rotation=20)

# Margin by Market
ax = axes[1, 0]
colors_mg = [GREEN if m > 0 else RED for m in market_df["margin_pct"]]
bars = ax.bar(market_df["market"], market_df["margin_pct"],
              color=colors_mg, width=0.6)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
for bar, val in zip(bars, market_df["margin_pct"]):
    offset = 0.3 if val >= 0 else -1.2
    ax.text(bar.get_x() + bar.get_width()/2,
            val + offset,
            f"{val:.1f}%", ha="center", fontsize=9, fontweight="bold")
ax.set_ylabel("Profit Margin (%)")
ax.set_title("Profit Margin by Market", fontweight="bold")
ax.tick_params(axis="x", rotation=20)

# Avg Order Value by Market
ax = axes[1, 1]
bars = ax.bar(market_df["market"], market_df["avg_order_val"],
              color=BLUE, alpha=0.8, width=0.6)
for bar, val in zip(bars, market_df["avg_order_val"]):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5,
            f"${val:.0f}", ha="center", fontsize=9, fontweight="bold")
ax.set_ylabel("Avg Order Value (USD)")
ax.set_title("Avg Order Value by Market", fontweight="bold")
ax.tick_params(axis="x", rotation=20)

plt.suptitle("Chart 16 — Market Performance Dashboard",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_16_market_performance.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print("   Saved: chart_16_market_performance.png")

# ── CHART 17: Region & Category Heatmap ───────────────────────────────────────
print("[Chart 17] Region x Category Heatmap...")

region_cat = df.groupby(["region", "category"]).agg(
    total_profit = ("profit", "sum"),
    margin_pct   = ("profit", lambda x: x.sum() / df.loc[x.index, "sales"].sum() * 100),
).reset_index()

# Pivot cho heatmap
pivot_profit = region_cat.pivot(index="region",
                                 columns="category",
                                 values="margin_pct")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    pivot_profit,
    annot=True,
    fmt=".1f",
    cmap="RdYlGn",
    center=0,
    ax=ax,
    cbar_kws={"label": "Profit Margin (%)"},
    linewidths=0.5,
    annot_kws={"size": 10},
)
ax.set_xlabel("Category", fontsize=12)
ax.set_ylabel("Region", fontsize=12)
ax.set_title("Chart 17 — Profit Margin Heatmap\n"
             "Region × Category (Red = Loss, Green = Profit)",
             fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_17_region_category_heatmap.png"),
            dpi=150)
plt.close()
print("   Saved: chart_17_region_category_heatmap.png")

# ── CHART 18: Top 10 Cities ───────────────────────────────────────────────────
print("[Chart 18] Top 10 Cities...")

city_df = df.groupby(["city", "country"]).agg(
    total_sales  = ("sales",  "sum"),
    total_profit = ("profit", "sum"),
    orders       = ("order_id","nunique"),
).reset_index()
city_df["margin_pct"] = city_df["total_profit"] / city_df["total_sales"] * 100
city_df["label"]      = city_df["city"] + "\n(" + city_df["country"] + ")"

top10_city    = city_df.nlargest(10,  "total_sales")
bottom5_city  = city_df.nsmallest(5, "total_profit")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

bars1 = ax1.barh(top10_city["label"], top10_city["total_sales"]/1e3,
                 color=BLUE, height=0.6)
for bar, row in zip(bars1, top10_city.itertuples()):
    ax1.text(bar.get_width() + 5,
             bar.get_y() + bar.get_height()/2,
             f"${row.total_sales/1e3:.0f}K | {row.margin_pct:.1f}%",
             va="center", fontsize=8.5)
ax1.set_xlabel("Total Sales (K USD)")
ax1.set_title("Top 10 Cities by Revenue", fontweight="bold")
ax1.set_xlim(0, top10_city["total_sales"].max()/1e3 * 1.4)

bars2 = ax2.barh(bottom5_city["label"], bottom5_city["total_profit"]/1e3,
                 color=RED, height=0.6)
for bar, row in zip(bars2, bottom5_city.itertuples()):
    ax2.text(bar.get_width() - 0.5,
             bar.get_y() + bar.get_height()/2,
             f"${row.total_profit/1e3:.1f}K",
             va="center", ha="right", color="white", fontsize=9)
ax2.axvline(0, color="black", linewidth=1)
ax2.set_xlabel("Total Profit (K USD)")
ax2.set_title("Bottom 5 Cities (Biggest Losses)", fontweight="bold")

plt.suptitle("Chart 18 — City-level Analysis",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CHART_DIR, "chart_18_city_analysis.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print("   Saved: chart_18_city_analysis.png")

# ── Export CSV cho Power BI ───────────────────────────────────────────────────
print("\n[Export] geo_summary.csv cho Power BI...")
csv_path = os.path.join(DATA_DIR, "geo_summary.csv")
country_df.round(2).to_csv(csv_path, index=False)
print(f"   Saved: {csv_path} ({len(country_df)} countries)")

# ── KEY INSIGHTS ──────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("KEY INSIGHTS - GEOGRAPHIC ANALYSIS")
print("=" * 60)

top1    = country_df.nlargest(1, "total_profit").iloc[0]
loss_c  = country_df[country_df["is_loss"]]
top_mkt = market_df.nlargest(1, "total_profit").iloc[0]
bot_mkt = market_df.nsmallest(1, "total_profit").iloc[0]

print(f"""
Country Analysis ({len(country_df)} countries):
  Most profitable : {top1['country']} 
                    Revenue ${top1['total_sales']/1e3:.0f}K | 
                    Profit  ${top1['total_profit']/1e3:.0f}K |
                    Margin  {top1['margin_pct']:.1f}%
  Loss-making     : {len(loss_c)} countries
  Top loss country: {country_df.nsmallest(1,'total_profit').iloc[0]['country']}
                    Profit ${country_df.nsmallest(1,'total_profit').iloc[0]['total_profit']/1e3:.1f}K

Market Analysis ({len(market_df)} markets):
  Best market  : {top_mkt['market']} 
                 Revenue ${top_mkt['total_sales']/1e6:.1f}M | 
                 Margin  {top_mkt['margin_pct']:.1f}%
  Worst market : {bot_mkt['market']}
                 Margin  {bot_mkt['margin_pct']:.1f}%

Top 5 Countries by Revenue:
""")
for _, r in country_df.nlargest(5, "total_sales").iterrows():
    print(f"  {r['country']:20s} ${r['total_sales']/1e3:7.0f}K sales | "
          f"{r['margin_pct']:5.1f}% margin")

print(f"""
Recommendation:
  Focus marketing budget on top 5 countries (highest ROI).
  Investigate loss-making countries — consider discount reduction.
  APAC & US markets show strongest growth potential.
""")
print(f" Charts: {CHART_DIR}")
print(f" CSV   : {csv_path}")
print("\n" + "=" * 60)
print("HOAN THANH Notebook 04 - Geographic Analysis!")
print("Buoc tiep theo: 02_Python/05_Forecast.py")
print("=" * 60)

GLOBAL SUPERSTORE - GEOGRAPHIC ANALYSIS

Data loaded: 51,290 rows | 147 countries

Total countries analyzed: 149
Profitable countries    : 119
Loss-making countries   : 30

[Chart 15] Top & Bottom Countries by Profit...
   Saved: chart_15_country_profit.png
[Chart 16] Market Performance...
   Saved: chart_16_market_performance.png
[Chart 17] Region x Category Heatmap...
   Saved: chart_17_region_category_heatmap.png
[Chart 18] Top 10 Cities...
   Saved: chart_18_city_analysis.png

[Export] geo_summary.csv cho Power BI...
   Saved: C:\DA\global-superstore-analysis\data\geo_summary.csv (149 countries)

KEY INSIGHTS - GEOGRAPHIC ANALYSIS

Country Analysis (149 countries):
  Most profitable : United States 
                    Revenue $2297K | 
                    Profit  $286K |
                    Margin  12.5%
  Loss-making     : 30 countries
  Top loss country: Turkey
                    Profit $-98.4K

Market Analysis (7 markets):
  Best market  : APAC 
                 Revenue $3.6M 